In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

y = train[TARGET]
X = train.drop(columns=[TARGET])

# find categorical columns
cat_features = X.select_dtypes("object").columns.tolist()

model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    eval_metric="AUC",
    loss_function="Logloss",
    task_type="GPU",
    devices="0",
    random_seed=42,
    early_stopping_rounds=200
)

model.fit(
    X,
    y,
    cat_features=cat_features,
    verbose=200
)

pred_probs = model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpu.csv", index=False)

/tmp/ipykernel_166175/2584916530.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes("object").columns.tolist()
Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 174ms	remaining: 14m 31s
200:	total: 29.5s	remaining: 11m 45s
400:	total: 58.6s	remaining: 11m 12s
600:	total: 1m 28s	remaining: 10m 48s
800:	total: 1m 58s	remaining: 10m 22s
1000:	total: 2m 28s	remaining: 9m 54s
1200:	total: 2m 59s	remaining: 9m 26s
1400:	total: 3m 28s	remaining: 8m 56s
1600:	total: 3m 58s	remaining: 8m 26s
1800:	total: 4m 28s	remaining: 7m 57s
2000:	total: 4m 58s	remaining: 7m 27s
2200:	total: 5m 28s	remaining: 6m 57s
2400:	total: 5m 58s	remaining: 6m 27s
2600:	total: 6m 28s	remaining: 5m 58s
2800:	total: 6m 58s	remaining: 5m 28s
3000:	total: 7m 28s	remaining: 4m 58s
3200:	total: 7m 58s	remaining: 4m 28s
3400:	total: 8m 28s	remaining: 3m 59s
3600:	total: 8m 58s	remaining: 3m 29s
3800:	total: 9m 28s	remaining: 2m 59s
4000:	total: 9m 58s	remaining: 2m 29s
4200:	total: 10m 28s	remaining: 1m 59s
4400:	total: 10m 58s	remaining: 1m 29s
4600:	total: 11m 29s	remaining: 59.8s
4800:	total: 11m 59s	remaining: 29.8s
4999:	total: 12m 29s	remaining: 0us


## Catboost Optimization

In [3]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])


y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes("object").columns.tolist()


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,

    eval_metric="AUC",
    loss_function="Logloss",

    task_type="GPU",
    devices="0",

    random_seed=42,

    early_stopping_rounds=200,

    verbose=200
)


model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features
)


pred_probs = model.predict_proba(test)[:, 1]


submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpuV1.1.csv", index=False)

print("submission_catboost_gpuV1.1.csv created")

/tmp/ipykernel_201766/3550931150.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes("object").columns.tolist()
Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8969475	best: 0.8969475 (0)	total: 127ms	remaining: 10m 35s
200:	test: 0.9140522	best: 0.9140522 (200)	total: 25.1s	remaining: 9m 58s
400:	test: 0.9150567	best: 0.9150567 (400)	total: 49.7s	remaining: 9m 29s
600:	test: 0.9155694	best: 0.9155699 (599)	total: 1m 14s	remaining: 9m 7s
800:	test: 0.9157893	best: 0.9157893 (800)	total: 1m 40s	remaining: 8m 44s
1000:	test: 0.9159017	best: 0.9159017 (1000)	total: 2m 5s	remaining: 8m 21s
1200:	test: 0.9159759	best: 0.9159759 (1200)	total: 2m 30s	remaining: 7m 56s
1400:	test: 0.9160285	best: 0.9160285 (1400)	total: 2m 55s	remaining: 7m 32s
1600:	test: 0.9160227	best: 0.9160461 (1497)	total: 3m 21s	remaining: 7m 7s
bestTest = 0.9160461128
bestIteration = 1497
Shrink model to first 1498 iterations.
submission_catboost_gpuV1.1.csv created


## Catboost with Stratifier K-fold Cross Validation

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
cat_features = X.select_dtypes(include=["object","string"]).columns.tolist()


N_FOLDS = 5

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== Fold {fold+1} =====")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(

        iterations=5000,
        learning_rate=0.03,
        depth=8,

        eval_metric="AUC",
        loss_function="Logloss",

        task_type="GPU",
        devices="0",

        random_seed=42,

        early_stopping_rounds=200,

        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features
    )

    val_pred = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(test)[:,1] / N_FOLDS

y_binary = y.map({"No":0,"Yes":1})
auc = roc_auc_score(y_binary, oof_preds)

print("\nCV ROC-AUC:", auc)

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

submission.to_csv("submission_catboost_cvV1.2.csv", index=False)

print("\nsubmission_catboost_cvV1.2.csv created")


===== Fold 1 =====


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8950917	best: 0.8950917 (0)	total: 131ms	remaining: 10m 54s
200:	test: 0.9133993	best: 0.9133993 (200)	total: 25.1s	remaining: 10m
400:	test: 0.9143979	best: 0.9143979 (400)	total: 49.8s	remaining: 9m 31s
600:	test: 0.9149137	best: 0.9149137 (600)	total: 1m 14s	remaining: 9m 7s
800:	test: 0.9151786	best: 0.9151786 (800)	total: 1m 39s	remaining: 8m 42s
1000:	test: 0.9153338	best: 0.9153342 (999)	total: 2m 4s	remaining: 8m 18s
1200:	test: 0.9154107	best: 0.9154107 (1200)	total: 2m 29s	remaining: 7m 53s
1400:	test: 0.9154507	best: 0.9154518 (1386)	total: 2m 54s	remaining: 7m 29s
1600:	test: 0.9154544	best: 0.9154636 (1537)	total: 3m 20s	remaining: 7m 4s
bestTest = 0.9154635668
bestIteration = 1537
Shrink model to first 1538 iterations.

===== Fold 2 =====


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8968641	best: 0.8968641 (0)	total: 130ms	remaining: 10m 48s
200:	test: 0.9145955	best: 0.9145955 (200)	total: 25.4s	remaining: 10m 5s
400:	test: 0.9155526	best: 0.9155526 (400)	total: 49.9s	remaining: 9m 32s
600:	test: 0.9160088	best: 0.9160088 (600)	total: 1m 14s	remaining: 9m 8s
800:	test: 0.9162528	best: 0.9162528 (800)	total: 1m 39s	remaining: 8m 44s
1000:	test: 0.9163851	best: 0.9163851 (1000)	total: 2m 5s	remaining: 8m 20s
1200:	test: 0.9164725	best: 0.9164756 (1191)	total: 2m 30s	remaining: 7m 55s
1400:	test: 0.9165164	best: 0.9165174 (1399)	total: 2m 55s	remaining: 7m 31s
1600:	test: 0.9165503	best: 0.9165503 (1600)	total: 3m 21s	remaining: 7m 6s
1800:	test: 0.9165631	best: 0.9165642 (1786)	total: 3m 46s	remaining: 6m 42s
2000:	test: 0.9165630	best: 0.9165764 (1834)	total: 4m 11s	remaining: 6m 17s
bestTest = 0.9165763557
bestIteration = 1834
Shrink model to first 1835 iterations.

===== Fold 3 =====


...

0:	test: 0.8961851	best: 0.8961851 (0)	total: 126ms	remaining: 10m 32s
200:	test: 0.9138825	best: 0.9138825 (200)	total: 25.3s	remaining: 10m 4s
400:	test: 0.9147505	best: 0.9147505 (400)	total: 50.3s	remaining: 9m 36s
600:	test: 0.9152471	best: 0.9152473 (599)	total: 1m 15s	remaining: 9m 12s
800:	test: 0.9154564	best: 0.9154565 (799)	total: 1m 40s	remaining: 8m 49s
1000:	test: 0.9155551	best: 0.9155583 (973)	total: 2m 6s	remaining: 8m 24s
1200:	test: 0.9156150	best: 0.9156151 (1194)	total: 2m 31s	remaining: 8m
1400:	test: 0.9156355	best: 0.9156366 (1353)	total: 2m 56s	remaining: 7m 34s
1600:	test: 0.9156465	best: 0.9156465 (1600)	total: 3m 22s	remaining: 7m 9s
1800:	test: 0.9156530	best: 0.9156600 (1766)	total: 3m 47s	remaining: 6m 43s
bestTest = 0.9156600237
bestIteration = 1766
Shrink model to first 1767 iterations.

===== Fold 4 =====


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8970565	best: 0.8970565 (0)	total: 131ms	remaining: 10m 53s
200:	test: 0.9149528	best: 0.9149528 (200)	total: 25.2s	remaining: 10m 1s
400:	test: 0.9159105	best: 0.9159108 (399)	total: 49.9s	remaining: 9m 32s
600:	test: 0.9164107	best: 0.9164107 (600)	total: 1m 15s	remaining: 9m 10s
800:	test: 0.9166485	best: 0.9166485 (800)	total: 1m 40s	remaining: 8m 47s
1000:	test: 0.9167580	best: 0.9167642 (988)	total: 2m 6s	remaining: 8m 23s
1200:	test: 0.9167765	best: 0.9167974 (1115)	total: 2m 31s	remaining: 7m 59s
bestTest = 0.9167973697
bestIteration = 1115
Shrink model to first 1116 iterations.

===== Fold 5 =====


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8950625	best: 0.8950625 (0)	total: 127ms	remaining: 10m 34s
200:	test: 0.9121724	best: 0.9121724 (200)	total: 25.2s	remaining: 10m 2s
400:	test: 0.9131918	best: 0.9131923 (399)	total: 50.1s	remaining: 9m 34s
600:	test: 0.9137147	best: 0.9137147 (600)	total: 1m 15s	remaining: 9m 11s
800:	test: 0.9139673	best: 0.9139675 (799)	total: 1m 40s	remaining: 8m 48s
1000:	test: 0.9140963	best: 0.9140965 (998)	total: 2m 6s	remaining: 8m 24s
1200:	test: 0.9141701	best: 0.9141701 (1200)	total: 2m 31s	remaining: 8m
1400:	test: 0.9141823	best: 0.9141901 (1360)	total: 2m 57s	remaining: 7m 35s
bestTest = 0.9141900837
bestIteration = 1360
Shrink model to first 1361 iterations.

CV ROC-AUC: 0.9157281988827864

submission_catboost_cvV1.2.csv created


## Catboost with Feature Engineering